#  Why are the simulation results to same across all generated models including the input?
The preliminary simulation results for growth rate are the same between all models including the input model in `comparing_models_0425.ipynb`. 

Ideas to check if they're actually the same:
1. check gene weights
2. check reaction weights
3. check reaction bounds
4. Print full integer length results of each simulation and compare (in code, not visually) that the numbers are identical

In [15]:
import pandas as pd

import os

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model, load_json_model
from cobra.summary import model_summary

In [16]:
# read in the csv files of the threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")

## Gene Weights Comparison

The notebook with the gene weights comparison can be found here: `/home/gmvaz/2026_GEMs/stm_model_test/comparing_models_0513.ipynb`

After normalizing the gene expression data, the genes were assigned "weights" to represent three kinds of expression state: (-1) downregulated, (0) neutral, and (+1) upregulated. I created three gene expression categories for iMAT implementation:

(1) 10% downregulated, 80% neutral, 10% upregulated, 
(2) 15% downregulated, 70% neutral, 15% upregulated, 
(3) 25% downregulated, 50% neutral, 25% upregulated. 

 

The preliminary model growth rate predictions from `comparing_models_0425.ipynb` were the same for all models. This is unexpected given that each model has (1) genes classified into varied states depending on the threshold category and (2) a different sum of genes in each expression state depending on the threshold category. 

I'll start with a top - down approach to evaluate if the models are different and "where" they differ. Let's first compare the gene weights between two threshold categories to determine if the gene weight of an individual gene changes between each category.

In [3]:
# Let's make a combined table of gene weights from the stm_threshold_1090 csv and the stm_threshold_1585 csv
geneweights_1090_1585 = stm_threshold_10_90.merge(stm_threshold_15_85, on = "genes", how = "left", suffixes=("_1090", "_1585"))

# Let's filter the table to only show genes whose gene weight changed based on the threshold category
geneweights_1090_1585[geneweights_1090_1585.threshold_category_1090 != geneweights_1090_1585.threshold_category_1585]

,genes,threshold_category_1090,threshold_category_1585
13,STM0014,0,-1
21,STM0022,0,-1
24,STM0025,0,-1
26,STM0027,0,-1
27,STM0028,0,-1
...,...,...,...
4530,STM4562,0,1
4541,STM4573,0,-1
4549,STM4581,0,1
4554,STM4586,0,1


## Reaction Weights Comparison

The different gene expression states and amounts of genes in each threshold category confirms that I correctly categorized gene weights for iMATpy. The next step is converting the gene weights to reaction weights.
The conversion process is as follows: 

(1) make a list of gene names from the gene weights file, 
(2) find the genes that overlap in both the input model and the gene weights file, 
(3) find genes that are in the input model but not in the gene weights file (if any), and
(4) make a new list with all the genes from the input model with it's associated gene weight and includes genes from the input model that were not found in the gene weights file. 

We include all the genes from the input model because when integrating the gene expression data to the model, we must have gene weights for every gene in the model. Genes found in the input model and not in the gene weights file are assigned a gene weight of (0) or neutral state since we don't know whether it's overexpressed or underexpressed. 

The new list now has all the genes found the input model with gene weight values. The `gene_to_rxn_weights()` funciton in iMATpy converts gene weights to reaction weights using the model's GPR rules. Gene-Protein-Reaction (GPR) rules uses Boolean logic (and/or) to describe how gene products come together to catalze reactions. 

### Why convert to reaction weights? 
Metabolic models are a network that a list of biochemical reactions, they aren't a list of genes. A reaction has reactants, products, flux, direction, and bounds (upper/lower). When run methods like iMAT or FBA, we're asking, "How much flux is going through the network (each reaction)?". One way to think about it is that genes exist like "rules" attached to a reaction. They say which gene encodes for the enzyme needed to catalyze a reaction and gene expression data provides evidence about a particular enzyme's activity level. What iMAT does at a high-level is
attempt to keep a reaction active if it has a positive reaction weight, inactivate a reaction with a negative weight if the organism can still grow or penalize growth if the reaction is needed, and remain flexible with reactions weighted neutral. Reaction weights qualitatively "tell" the iMAT algorithm if there's evidence to suggest whether a reaction is active, inactive, or unaltered. With this information, the algorithm then finds a functional flux distribution that agrees with the expression data as best as possible while staying functional. "Feasible" in this case means that these constraints are satisfied: 

(1) mass balance
(2) reaction bounds
(3) directionality
(4) nutrient constraints
(5) not necessary but any required functions (some task like biomass or ATP production must carry some minimum flux)

In [4]:
stm1090_vs_1585_rxnweights = pd.read_csv('stm1090_vs1585_rxnweights.csv')

In [5]:
stm1090_vs_1585_rxnweights

,Unnamed: 0,a,b
0,ACNAMt2pp,0.0,1.0
1,ALLTN,0.0,1.0
2,AOXSr,0.0,-1.0
3,ARAI,0.0,1.0
4,ARBt2rpp,0.0,1.0
...,...,...,...
122,OCDCEAt2pp,0.0,1.0
123,MUREINLPPTP,0.0,1.0
124,2HH24DDH1,0.0,-1.0
125,GALCTND,0.0,1.0


## FBA Growth Predictions Comparison

In [6]:
# Set the Gurobi license and solver
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [7]:
# load the input S.Tm LT2 GEM
base_model = load_model("STM_v1_0")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [8]:
# load the stm models generated in `comparing_model_0513.ipynb`
stm1090_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1090_imat_restrictions.json")
stm1585_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1585_imat_restrictions.json")
stm2575_model = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm2575_imat_restrictions.json")

In [9]:
fba_stm1090 = stm1090_model.optimize()
fba_stm1585 = stm1585_model.optimize()
fba_stm2575 = stm2575_model.optimize()
fba_base = base_model.optimize()

In [10]:
stm1090_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.002263,0,0.00%
cl_e,EX_cl_e,0.002263,0,0.00%
cobalt2_e,EX_cobalt2_e,0.001509,0,0.00%
cu2_e,EX_cu2_e,0.001509,0,0.00%
fe2_e,EX_fe2_e,0.003502,0,0.00%
glc__D_e,EX_glc__D_e,5,6,100.00%
k_e,EX_k_e,0.08486,0,0.00%
mg2_e,EX_mg2_e,0.003772,0,0.00%
mn2_e,EX_mn2_e,0.001509,0,0.00%
mobd_e,EX_mobd_e,0.001509,0,0.00%


In [11]:
print(fba_base.objective_value)
print(fba_stm1090.objective_value)
print(fba_stm1585.objective_value)
print(fba_stm2575.objective_value)

0.4778336607607207
0.47783366076072126
0.47783366076072126
0.47783366076072126


### Making sense of the predicted growth rates

What is growth rate in FBA? What does the number mean biologically? What growth rate values should we expect? How can we compare between values?

Let's use the predicted growth rate of the input model as a control to gauge how different the predictions are for the extracted models. I'll divide the growth rate of the input model by the growth rate of each extracted model to see the percent change.


#### Calculating percent decrease and percent retained

Inspect the variables I'm using first. 
```
type(fba_base.objective_value) = float
dir(fba_base.objective_value)
print(fba_base.objective_value) = 0.4778336607607207
```
From this I know that my objective values are float (numbers with a decimal point) and I know that when I print it out, I get a long decimal. 

I want to (1) have a math equation that solves the difference of extracted model and the base model and (2) takes the difference and calculates what percentage it is of the base model. The final answer should be to the second decimal point. 

First, I'll calculate percent difference for each extracted model: 

percent_decrease = ((input_model - extracted_model) / input_model) * 100

In [12]:
stm1090_decrease = ((fba_base.objective_value - fba_stm1090.objective_value) / (fba_base.objective_value))
print(stm1090_decrease)

# The value assigned to the variable `stm1090_decrease` is a float. 
# I need to multiply by 100, convert to a percent, and round to the second decimal point. 

#  `format()` is a built in python function that converts the value to a string-representation of a specified format. 
# in this case, I am turning a decimal number into a percentage string with two digits after the decimal
# `format(float, ".2%") - `float` is the number that I want to change. "%" is saying take this decimal number, multiply it by 100,
# and add a percent sign. The `.2` is saying round to the second decimal point. 

stm1090_percentdecrease = format(stm1090_decrease, ".2%")
print(stm1090_percentdecrease)

stm1585_decrease = ((fba_base.objective_value - fba_stm1585.objective_value) / (fba_base.objective_value))
print(stm1585_decrease)
stm1585_percentdecrease = format(stm1585_decrease, ".2%")
print(stm1585_percentdecrease)

stm2575_decrease = ((fba_base.objective_value - fba_stm2575.objective_value) / (fba_base.objective_value))
print(stm2575_decrease)
stm2575_percentdecrease = format(stm2575_decrease, ".2%")
print(stm2575_percentdecrease)

-1.1617254243429182e-15
-0.00%
-1.1617254243429182e-15
-0.00%
-1.1617254243429182e-15
-0.00%


I can also calculate the percent of growth retained from the input model:

percent retained = (extracted_model / input_model) * 100

In [13]:
stm1090_percentretained = fba_stm1090.objective_value / fba_base.objective_value
print(format(stm1090_percentretained, ".2%"))

stm1585_percentretained = fba_stm1585.objective_value / fba_base.objective_value
print(format(stm1585_percentretained, ".2%"))

stm2575_percentretained = fba_stm2575.objective_value / fba_base.objective_value
print(format(stm2575_percentretained, ".2%"))

100.00%
100.00%
100.00%


### Thoughts

Models `stm1090_model` and `stm1585_model` have predicted growth rates that are almost the same. They have a predicted growth rate of 14.78% and 14.93% lower than the input model's growth rate, respecitively. This suggests that threshold categories of 10%/90% and 15%/85% do not make a noticeable impact on overall growth rate predictions given these constraints. Although, it is possible that the predicted growth rate could noticeably change once the environmental conditions are altered. 

### Moving Forward

We now have generated models that integrate gene expression data correctly and have growth rate predictions that vary based on the threshold category and are reasonable. 

Gene expression data of S. Tm infecting mice with and without other microbes present has been integrated into the extracted GEMs. But the GEMs do not have environmental constraints to mimic the gut under colitis conditions. I also do not know what the objective function is for S. Tm to grow in the model and what the preassigned environmental constraints are. These are important to know so that I can interpret the FBA results in context of the environment S. Tm is "living in". The FBA results alone are not relatable to our experimental questions. The **goal** now is to design the environmental conditions of S. Tm in a colitis environment so that the results from FBA simulations can be interpreted in that context. 

## What's an objective function? 

Objective functions are used to search over a possible solution space to find particular functional states. The mathematical form of the objective function is: 
Z = {**w,v**}
where **w** is a vector of weights that tells the model which combination of reactions matter for the goal, and **v** is the vector of fluxes through reactions in the model. We get to choose vector **w** based on the biological goal we want: biomass production, ATP production, some metabolite secretion, or flux through a specific reaction. This function can be an internal biological reaction in the model or an equation estimating a biological objective. Z is the funcitonality the model is trying to maximize. 

## What objective functions are available for S. Tm? 

From reading papers on FBA, I was under the impression that people "solve" for appropriate cell objectives and that these were "options" in the GEMs. The S. Tm `input_model` does not have other objective options which leads me to think that you can set any reaction to be the objective. That is to say, you can maximize flux through any desired reaction. 

#### Thoughts
- It would be interesting to set the ATP maintenance reaction as the objective and compare to the biomass reaction results.
- Two studies concluded that the optimization of biomass production or growth is best fit for prediction growth data for E. coli and S. cerevisiae. 
- We don't fully know what are the cellular objectives that drive evolution of phenotypic states. 
- **The objective function are likely-condition dependent and change with adaptation**
	- Studies suggesting this: (372, 386, and 339) in the book. 

**Takeaways**
It's an interesting aspect to COBRA modeling because the objective function defines the metabolic selection process and has implications to the evolutionary history and adaptation strategies of that organism. 

[The Biomass Objective Function: Feist and Palsson, 2010:](https://www.sciencedirect.com/science/article/pii/S1369527410000512?via%3Dihub)
When choosing an objective function the key question to ask is,  `What does a microorganism try to do in a given environment?`. We don't often (if ever) know the answer to this question because it would entail knowing the evolutionary history of the microorganism. 

[Here's](http://bigg.ucsd.edu/universal/reactions/BIOMASS_iRR1083_metals) where to see the objective reaction for S. Tm. 

In [14]:
# Let's look at the objective in each model - they should be the same because I did not change them.
# The objective should be to maximize biomass. 
print(base_model.objective)
print(stm1090_model.objective)
print(stm1585_model.objective)
print(stm2575_model.objective)

Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Maximize
1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4


## How do we simulate environmental conditions in a GEM??

### What are environmental constraints? 
#### Find the minimal media uptake rates in stm_v1_0

The methods section of [(Seif et al., 2018)](https://www.nature.com/articles/s41467-018-06112-5) states that the stm_v1_0 model has uptake rates for in vitro minimal media. Let's first find these. 

**Coding notes** 
I searched the Cobrapy API for `uptake rates` and the cobra.medium module seems promising.

In API, functions are written like this:

  ```
  cobra.medium.find_boundary_types(model: cobra.Model, boundary_type: str, external_compartment: str | None = None) → List[cobra.Reaction]
  ```

and can be read like: 

library: cobra
module: medium
function name: find_boundary_types

parameters, aka inputs (the value passed through a parameter is an argument):
- model
- boundary_type
- external_compartment, optional

output:
- list of cobra Reaction objects

In [17]:
import sys
sys.version

'3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]'